# Intrinsic semiconductor statistics from a 2D checkerboard potential

**Status:** Computational laboratory using the maintained PhysKit two-dimensional checkerboard Bloch Hamiltonian. This is an idealized model, not a silicon band-structure calculation.

**Prerequisite:** [Two-dimensional checkerboard-potential Bloch bands](../../../../docs/lecturenotes/solidstate/bloch2d/checkerboard/index.md)

The notebook solves

$$
V(x,y)=V_0\cos\left(\frac{2\pi x}{a}\right)\cos\left(\frac{2\pi y}{a}\right)
$$

in a reciprocal plane-wave basis, verifies a complete gap over the full Brillouin zone, constructs the 2D density of states, and solves intrinsic charge neutrality. Carrier densities are reported per unit **area**. They are not interchangeable with the 1D linear densities or the 3D bulk concentrations in Sze--Ng.

Reference for the equilibrium intrinsic-semiconductor framework: S. M. Sze and K. K. Ng, *Physics of Semiconductor Devices*, 3rd ed. (Wiley, 2007). The exposition and code are original and contain no reproduced textbook excerpts.


## Learning objectives

1. Construct and solve a genuine 2D checkerboard Bloch Hamiltonian with PhysKit.
2. Follow bands along $\Gamma-X-M-\Gamma$ and independently scan the complete zone.
3. Identify the first isolated two-band valence manifold and its indirect gap.
4. Check reciprocal-basis convergence and normalized 2D DOS state counts.
5. Solve exact Fermi--Dirac intrinsic neutrality, $n_{2D}=p_{2D}$.
6. Compare the exact result with a nondegenerate band-integrated 2D approximation and calculate sheet conductance when mobilities are supplied.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.constants import Boltzmann, electron_volt, elementary_charge
from scipy.optimize import brentq
from scipy.special import expit

from projectkoios.physkit.periodic.kpoints import KPointPath2D
from projectkoios.physkit.solidstate.bloch2d.checkerboard.model import CheckerboardPotentialBloch2D
from projectkoios.physkit.solidstate.semiconductors.effmass_2d.cell import PeriodicPrimitiveCell2D
from projectkoios.physkit.units import MatrixQuantity, ScalarQuantity, UnitSystem

K_B_EV = Boltzmann / electron_volt
SPIN_DEGENERACY = 2.0


## 1. Checkerboard model and numerical representation

The displayed square cell has basis $a\mathbf e_x,a\mathbf e_y$. Because the product potential is also invariant under simultaneous half-cell translations, the square display cell contains two translation-related sectors. The first isolated valence manifold therefore contains two energy-ordered bands. With spin degeneracy, filling this manifold corresponds to four electrons per displayed square cell.

The reciprocal basis uses modes $-M\leq m_x,m_y\leq M$. The default $V_0=4$ eV is chosen to produce a visible complete gap in this idealized model; it is not a fitted material parameter.


In [ ]:
lattice_constant_angstrom = 5.43
checkerboard_amplitude_ev = 4.0
reciprocal_mode_limit = 3
band_count = 8
valence_band_count = 2
full_zone_axis_count = 32

cell = PeriodicPrimitiveCell2D(
    primitive_basis=MatrixQuantity(
        magnitude=lattice_constant_angstrom * np.eye(2),
        unit=UnitSystem.METAL.length_unit,
    ),
    unit_system=UnitSystem.METAL,
)
model = CheckerboardPotentialBloch2D(
    cell=cell,
    amplitude=ScalarQuantity(
        checkerboard_amplitude_ev,
        UnitSystem.METAL.energy_unit,
    ),
)
reciprocal_unit = cell.reciprocal_basis.unit
zone_edge = np.pi / lattice_constant_angstrom


In [ ]:
coordinate = np.linspace(
    -0.5 * lattice_constant_angstrom,
    0.5 * lattice_constant_angstrom,
    241,
)
x_grid, y_grid = np.meshgrid(coordinate, coordinate, indexing="ij")
potential_ev = checkerboard_amplitude_ev * np.cos(
    2.0 * np.pi * x_grid / lattice_constant_angstrom
) * np.cos(2.0 * np.pi * y_grid / lattice_constant_angstrom)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
contour = ax.contourf(x_grid, y_grid, potential_ev, levels=31, cmap="coolwarm")
fig.colorbar(contour, ax=ax, label="Potential (eV)")
ax.set(
    xlabel=r"$x$ (Å)",
    ylabel=r"$y$ (Å)",
    title="Alternating checkerboard potential",
    aspect="equal",
)
fig.tight_layout()


## 2. High-symmetry path and full-zone gap

A path plot is useful for interpretation but cannot establish a complete gap. We therefore use $\Gamma-X-M-\Gamma$ for visualization and a separate uniform two-dimensional mesh for all extrema, DOS, and carrier integrals.


In [ ]:
path = KPointPath2D(
    nodes=np.array(
        [
            [0.0, 0.0],
            [zone_edge, 0.0],
            [zone_edge, zone_edge],
            [0.0, 0.0],
        ]
    ),
    labels=(r"$\Gamma$", r"$X$", r"$M$", r"$\Gamma$"),
    points_per_segment=61,
)
path_result = model.solve_bands(
    bloch_wave_vectors=MatrixQuantity(path.values, reciprocal_unit),
    reciprocal_mode_limit=reciprocal_mode_limit,
    band_count=band_count,
)
path_bands_ev = path_result.energies.magnitude

fig, ax = plt.subplots(figsize=(7.6, 4.8))
for band_index in range(band_count):
    ax.plot(path.distances, path_bands_ev[:, band_index], linewidth=1.1)
for node_distance in path.node_distances:
    ax.axvline(node_distance, color="0.75", linewidth=0.8)
ax.set_xticks(path.node_distances, path.labels)
ax.set(
    ylabel="Energy (eV)",
    title="Checkerboard bands along the high-symmetry path",
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


In [ ]:
k_axis = np.linspace(
    -zone_edge,
    zone_edge,
    full_zone_axis_count,
    endpoint=False,
)
kx_grid, ky_grid = np.meshgrid(k_axis, k_axis, indexing="ij")
full_zone_vectors = np.column_stack((kx_grid.ravel(), ky_grid.ravel()))
full_zone_result = model.solve_bands(
    bloch_wave_vectors=MatrixQuantity(full_zone_vectors, reciprocal_unit),
    reciprocal_mode_limit=reciprocal_mode_limit,
    band_count=band_count,
)
full_zone_bands_ev = full_zone_result.energies.magnitude

valence_block_ev = full_zone_bands_ev[:, :valence_band_count]
conduction_block_ev = full_zone_bands_ev[:, valence_band_count:]
valence_edge_ev = float(np.max(valence_block_ev))
conduction_edge_ev = float(np.min(conduction_block_ev))
band_gap_ev = conduction_edge_ev - valence_edge_ev
valence_flat_index = int(np.argmax(valence_block_ev))
valence_k_index, valence_band_index = np.unravel_index(
    valence_flat_index, valence_block_ev.shape
)
conduction_flat_index = int(np.argmin(conduction_block_ev))
conduction_k_index, conduction_offset = np.unravel_index(
    conduction_flat_index, conduction_block_ev.shape
)
conduction_band_index = valence_band_count + conduction_offset

assert band_gap_ev > 0.0

pd.Series(
    {
        "valence bands in filled manifold": valence_band_count,
        "valence edge Ev (eV)": valence_edge_ev,
        "conduction edge Ec (eV)": conduction_edge_ev,
        "complete indirect gap Eg (eV)": band_gap_ev,
        "valence-edge band (1-based)": valence_band_index + 1,
        "conduction-edge band (1-based)": conduction_band_index + 1,
        "valence-edge kx (1/Å)": full_zone_vectors[valence_k_index, 0],
        "valence-edge ky (1/Å)": full_zone_vectors[valence_k_index, 1],
        "conduction-edge kx (1/Å)": full_zone_vectors[conduction_k_index, 0],
        "conduction-edge ky (1/Å)": full_zone_vectors[conduction_k_index, 1],
    },
    name="full-zone result",
)


In [ ]:
local_gap_ev = (
    full_zone_bands_ev[:, valence_band_count]
    - full_zone_bands_ev[:, valence_band_count - 1]
).reshape(full_zone_axis_count, full_zone_axis_count)

fig, ax = plt.subplots(figsize=(6.2, 5.1))
image = ax.pcolormesh(
    k_axis / zone_edge,
    k_axis / zone_edge,
    local_gap_ev.T,
    shading="auto",
)
fig.colorbar(image, ax=ax, label="Local band separation (eV)")
ax.set(
    xlabel=r"$k_x/(\pi/a)$",
    ylabel=r"$k_y/(\pi/a)$",
    title="Separation between bands 2 and 3",
    aspect="equal",
)
fig.tight_layout()


## 3. Reciprocal-basis convergence

The complete-zone mesh above establishes the discrete represented gap for $M=3$. The next check evaluates the identified $\Gamma$ valence edge and $M$ conduction edge as the reciprocal cutoff is increased. Agreement is numerical convergence evidence for those edges, not material validation.


In [ ]:
edge_vectors = np.array(
    [
        [0.0, 0.0],
        [-zone_edge, -zone_edge],
    ]
)
convergence_rows = []
for mode_limit in (2, 3, 4):
    edge_energies = model.solve_bands(
        bloch_wave_vectors=MatrixQuantity(edge_vectors, reciprocal_unit),
        reciprocal_mode_limit=mode_limit,
        band_count=4,
    ).energies.magnitude
    approximate_gap = (
        edge_energies[1, valence_band_count]
        - edge_energies[0, valence_band_count - 1]
    )
    convergence_rows.append(
        {
            "mode_limit": mode_limit,
            "basis_size": (2 * mode_limit + 1) ** 2,
            "Gamma valence edge (eV)": edge_energies[0, 1],
            "M conduction edge (eV)": edge_energies[1, 2],
            "edge gap (eV)": approximate_gap,
        }
    )
convergence_table = pd.DataFrame(convergence_rows)
assert abs(
    convergence_table.iloc[-1]["edge gap (eV)"]
    - convergence_table.iloc[-2]["edge gap (eV)"]
) < 2e-5
convergence_table


## 4. Two-dimensional density of states

For spin degeneracy $g_s=2$,

$$
g_{2D}(E)=\frac{g_s}{(2\pi)^2}\sum_n\int_{\mathrm{BZ}}
\delta(E-E_n(\mathbf k))\,d^2k.
$$

It has units of states per energy per area. Each complete band integrates to $g_s/a^2$ states per area. We use Gaussian broadening only for the plotted DOS; carrier integrals below use the unbroadened sampled band energies directly.


In [ ]:
broadening_ev = 0.035
energy_grid_ev = np.linspace(
    float(np.min(full_zone_bands_ev[:, :4])) - 0.2,
    float(np.max(full_zone_bands_ev[:, :4])) + 0.2,
    1600,
)
delta_k = 2.0 * zone_edge / full_zone_axis_count
state_weight_per_angstrom2 = (
    SPIN_DEGENERACY * delta_k**2 / (2.0 * np.pi) ** 2
)


def gaussian_dos_2d(selected_bands_ev):
    sampled_energies = selected_bands_ev.reshape(-1)
    differences = energy_grid_ev[:, None] - sampled_energies[None, :]
    kernels = np.exp(-0.5 * (differences / broadening_ev) ** 2)
    kernels /= np.sqrt(2.0 * np.pi) * broadening_ev
    return state_weight_per_angstrom2 * np.sum(kernels, axis=1)


valence_dos_per_ev_angstrom2 = gaussian_dos_2d(
    full_zone_bands_ev[:, :valence_band_count]
)
conduction_dos_per_ev_angstrom2 = gaussian_dos_2d(
    full_zone_bands_ev[:, valence_band_count : valence_band_count + 2]
)
expected_two_band_states_per_angstrom2 = (
    2.0 * SPIN_DEGENERACY / lattice_constant_angstrom**2
)
valence_states_per_angstrom2 = np.trapezoid(
    valence_dos_per_ev_angstrom2, energy_grid_ev
)
conduction_states_per_angstrom2 = np.trapezoid(
    conduction_dos_per_ev_angstrom2, energy_grid_ev
)
assert np.isclose(
    valence_states_per_angstrom2,
    expected_two_band_states_per_angstrom2,
    rtol=3e-3,
)
assert np.isclose(
    conduction_states_per_angstrom2,
    expected_two_band_states_per_angstrom2,
    rtol=3e-3,
)

fig, ax = plt.subplots(figsize=(7.3, 4.6))
ax.plot(energy_grid_ev, valence_dos_per_ev_angstrom2, label="two valence bands")
ax.plot(energy_grid_ev, conduction_dos_per_ev_angstrom2, label="two conduction bands")
ax.axvspan(valence_edge_ev, conduction_edge_ev, color="0.9", label="complete gap")
ax.set(
    xlabel="Energy (eV)",
    ylabel=r"$g_{2D}(E)$ (states eV$^{-1}$ Å$^{-2}$)",
    title=f"2D DOS (Gaussian sigma = {broadening_ev:.3f} eV)",
)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## 5. Exact 2D intrinsic charge neutrality

The full-zone sampled bands give

$$
n_{2D}=\frac{g_s}{(2\pi)^2}\sum_{c}\int_{\mathrm{BZ}}
f(E_c(\mathbf k),\mu,T)\,d^2k,
$$

$$
p_{2D}=\frac{g_s}{(2\pi)^2}\sum_{v}\int_{\mathrm{BZ}}
[1-f(E_v(\mathbf k),\mu,T)]\,d^2k.
$$

The intrinsic chemical potential is obtained from $n_{2D}=p_{2D}$. The result is a sheet density in cm$^{-2}$.


In [ ]:
def fermi_occupation(energy_ev, chemical_potential_ev, temperature_k):
    return expit(
        (chemical_potential_ev - energy_ev) / (K_B_EV * temperature_k)
    )


def carrier_densities_2d(chemical_potential_ev, temperature_k):
    valence_occupation = fermi_occupation(
        full_zone_bands_ev[:, :valence_band_count],
        chemical_potential_ev,
        temperature_k,
    )
    conduction_occupation = fermi_occupation(
        full_zone_bands_ev[:, valence_band_count:],
        chemical_potential_ev,
        temperature_k,
    )
    electrons_per_angstrom2 = state_weight_per_angstrom2 * np.sum(
        conduction_occupation
    )
    holes_per_angstrom2 = state_weight_per_angstrom2 * np.sum(
        1.0 - valence_occupation
    )
    return float(electrons_per_angstrom2), float(holes_per_angstrom2)


def intrinsic_state_2d(temperature_k):
    thermal_energy_ev = K_B_EV * temperature_k

    def imbalance(chemical_potential_ev):
        electrons, holes = carrier_densities_2d(
            chemical_potential_ev, temperature_k
        )
        return electrons - holes

    chemical_potential_ev = brentq(
        imbalance,
        valence_edge_ev - 20.0 * thermal_energy_ev,
        conduction_edge_ev + 20.0 * thermal_energy_ev,
        xtol=1e-13,
        rtol=1e-13,
    )
    electrons, holes = carrier_densities_2d(
        chemical_potential_ev, temperature_k
    )
    return {
        "temperature_K": float(temperature_k),
        "mu_i_eV": chemical_potential_ev,
        "n_2D_per_cm2": electrons * 1.0e16,
        "p_2D_per_cm2": holes * 1.0e16,
    }


In [ ]:
temperature_k = 300.0
intrinsic_2d = intrinsic_state_2d(temperature_k)
relative_neutrality_error = abs(
    intrinsic_2d["n_2D_per_cm2"] - intrinsic_2d["p_2D_per_cm2"]
) / max(intrinsic_2d["n_2D_per_cm2"], intrinsic_2d["p_2D_per_cm2"])
assert relative_neutrality_error < 2e-8

pd.Series(
    {**intrinsic_2d, "relative neutrality error": relative_neutrality_error},
    name="exact 2D intrinsic state",
)


## 6. Nondegenerate 2D effective densities

Rather than impose a parabolic effective mass at a degenerate edge, we integrate the calculated bands to define temperature-dependent nondegenerate density scales,

$$
N_C^{2D}=\frac{g_s}{(2\pi)^2}\sum_c\int
\exp\left[-\frac{E_c(\mathbf k)-E_C}{k_BT}\right]d^2k,
$$

with the analogous $N_V^{2D}$. Then

$$
n_i^{2D}=\sqrt{N_C^{2D}N_V^{2D}}
\exp\left(-\frac{E_g}{2k_BT}\right),
$$

$$
E_i=\frac{E_C+E_V}{2}+\frac{k_BT}{2}
\ln\left(\frac{N_V^{2D}}{N_C^{2D}}\right).
$$

This is the dimensionally appropriate 2D analogue of the familiar bulk nondegenerate construction. It is compared with, not substituted for, the exact Fermi--Dirac neutrality result.


In [ ]:
def nondegenerate_state_2d(temperature_k):
    thermal_energy_ev = K_B_EV * temperature_k
    nc_per_angstrom2 = state_weight_per_angstrom2 * np.sum(
        np.exp(
            -(full_zone_bands_ev[:, valence_band_count:] - conduction_edge_ev)
            / thermal_energy_ev
        )
    )
    nv_per_angstrom2 = state_weight_per_angstrom2 * np.sum(
        np.exp(
            -(valence_edge_ev - full_zone_bands_ev[:, :valence_band_count])
            / thermal_energy_ev
        )
    )
    intrinsic_density_per_angstrom2 = np.sqrt(
        nc_per_angstrom2 * nv_per_angstrom2
    ) * np.exp(-band_gap_ev / (2.0 * thermal_energy_ev))
    intrinsic_level_ev = (
        0.5 * (conduction_edge_ev + valence_edge_ev)
        + 0.5 * thermal_energy_ev * np.log(nv_per_angstrom2 / nc_per_angstrom2)
    )
    return {
        "Nc_2D_per_cm2": nc_per_angstrom2 * 1.0e16,
        "Nv_2D_per_cm2": nv_per_angstrom2 * 1.0e16,
        "mu_i_MB_eV": intrinsic_level_ev,
        "ni_2D_MB_per_cm2": intrinsic_density_per_angstrom2 * 1.0e16,
    }


nondegenerate_2d = nondegenerate_state_2d(temperature_k)
pd.Series(
    {
        **nondegenerate_2d,
        "exact ni_2D_per_cm2": intrinsic_2d["n_2D_per_cm2"],
        "MB_over_exact_ni": (
            nondegenerate_2d["ni_2D_MB_per_cm2"]
            / intrinsic_2d["n_2D_per_cm2"]
        ),
    },
    name="2D nondegenerate comparison at 300 K",
)


In [ ]:
# Optional illustrative room-temperature drift inputs.
electron_mobility_cm2_per_vs = 1350.0
hole_mobility_cm2_per_vs = 480.0
sheet_conductance_siemens = (
    elementary_charge
    * intrinsic_2d["n_2D_per_cm2"]
    * (electron_mobility_cm2_per_vs + hole_mobility_cm2_per_vs)
)
sheet_resistance_ohm_per_square = 1.0 / sheet_conductance_siemens

pd.Series(
    {
        "sheet conductance (S)": sheet_conductance_siemens,
        "sheet resistance (ohm/square)": sheet_resistance_ohm_per_square,
    },
    name="illustrative intrinsic sheet transport",
)


In [ ]:
temperatures_k = np.linspace(150.0, 600.0, 46)
exact_rows = [intrinsic_state_2d(value) for value in temperatures_k]
comparison_rows = []
for exact_row in exact_rows:
    approximate = nondegenerate_state_2d(exact_row["temperature_K"])
    comparison_rows.append({**exact_row, **approximate})
temperature_table = pd.DataFrame(comparison_rows)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.3))
axes[0].semilogy(
    temperature_table["temperature_K"],
    temperature_table["n_2D_per_cm2"],
    label="exact Fermi--Dirac",
)
axes[0].semilogy(
    temperature_table["temperature_K"],
    temperature_table["ni_2D_MB_per_cm2"],
    linestyle="--",
    label="nondegenerate",
)
axes[0].set(
    xlabel="Temperature (K)",
    ylabel=r"$n_i^{2D}$ (cm$^{-2}$)",
    title="Intrinsic sheet density",
)
axes[0].legend()
axes[1].plot(
    temperature_table["temperature_K"],
    temperature_table["mu_i_eV"],
    label="exact",
)
axes[1].plot(
    temperature_table["temperature_K"],
    temperature_table["mu_i_MB_eV"],
    linestyle="--",
    label="nondegenerate",
)
axes[1].axhline(valence_edge_ev, color="C2", linestyle=":", label=r"$E_V$")
axes[1].axhline(conduction_edge_ev, color="C3", linestyle=":", label=r"$E_C$")
axes[1].set(
    xlabel="Temperature (K)",
    ylabel="Energy (eV)",
    title="Intrinsic chemical potential",
)
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.25)
fig.tight_layout()


## 7. Interpretation and limitations

- This is a genuine nonseparable 2D checkerboard potential. It is not the separable egg-carton sum $\cos(Gx)+\cos(Gy)$.
- The first single-band separation is negative, while a positive complete gap appears after the first two bands. The electronic filling must therefore include the complete two-band valence manifold.
- The exact carrier calculation returns cm$^{-2}$. Converting it to cm$^{-3}$ requires an independently justified physical thickness; none is assumed here.
- The 3D Sze--Ng formulas for $N_C$, $N_V$, and bulk conductivity must not be applied directly to this 2D DOS. The notebook instead constructs dimensionally consistent 2D density scales.
- The sheet-transport estimate uses supplied mobilities and does not calculate scattering.
- No dopants, incomplete ionization, recombination-generation, electrostatics, contacts, many-body corrections, or real-material fitting are included.

### Exercises

1. Increase $M$ and the full-zone grid independently; distinguish basis error from integration error.
2. Find the critical $|V_0|$ at which the two-band valence manifold first becomes isolated.
3. Compare the product checkerboard with the separable egg-carton potential.
4. Derive a low-energy multivalley approximation near the $M$ conduction edge and compare its $N_C^{2D}$ with the full-band integral.
5. Change the sign of $V_0$ and confirm spectral invariance before interpreting translated real-space potentials.
